Daily Challenge: Pinecone Serverless Reranking in Action

Why are we doing this?
Reranking models boost search relevance by assigning similarity scores between a query and documents, then reordering results so the most pertinent information appears first. In contexts like healthcare, this helps clinicians quickly access the most critical clinical notes.

Task Overview & Detailed Explanations
Below is a skeleton pipeline. Each numbered item is an action you must complete. After every instruction, you’ll find a clear explanation of what to do and why it’s important. Whenever you see ..., replace it with the appropriate code or value, using the hint for guidance.

In [4]:
# Daily Challenge: Pinecone Serverless Reranking

# =============================================================================
# PARTIE 1: Load Documents & Execute Reranking Model
# =============================================================================

print("PARTIE 1: Configuration et Test du Reranking")
print("=" * 60)

# Étape 1: Installation des bibliothèques (à exécuter dans votre terminal)
!pip install pinecone==6.0.1 pinecone-notebooks transformers sentence-transformers

PARTIE 1: Configuration et Test du Reranking
  Using cached pinecone-6.0.1-py3-none-any.whl.metadata (8.8 kB)
  Using cached pinecone_notebooks-0.1.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl.metadata (1.2 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-

In [14]:
# Étape 2: Authentification avec Pinecone
import os
print("Étape 2: Authentification...")

if not os.environ.get("PINECONE_API_KEY"):
    # This part is for Colab environment and might require user interaction
    # or setting the API key in Colab Secrets as previously discussed.
    try:
        from pinecone_notebooks.colab import Authenticate
        Authenticate()
        print("Authentification réussie via pinecone_notebooks!")
    except ImportError:
        print("❌ 'pinecone_notebooks' not found. Please ensure it's installed.")
        print("Please set your Pinecone API key in Colab Secrets (the key icon on the left panel) with the name 'PINECONE_API_KEY').")
    except Exception as e:
        print(f"An error occurred during authentication: {e}")
else:
    print("Clé API déjà configurée dans l'environnement!")

Étape 2: Authentification...


Authentification réussie via pinecone_notebooks!


In [17]:
# Étape 4: Définir la requête et les documents
    # What to do: Replace ... with a list of five example sentences that include both references to the fruit “apple” and the company “Apple Inc.”.
    # Why: You need a small set of documents to test the reranker’s ability to distinguish between different contexts of the same word.

print("\n Étape 4: Préparation des documents de test...")

query = "Tell me about Apple's products"
documents = [
    "Apple is a delicious red fruit that grows on trees and is rich in vitamins",
    "Apple Inc. is a technology company that makes iPhones, MacBooks, and iPads",
    "The apple tree produces fruit in autumn seasons and requires specific climate",
    "Apple's latest iPhone features advanced camera technology and A17 chip",
    "Green apples are more tart than red apples and contain more antioxidants"
]

print(f"Requête: {query}")
print(f"Nombre de documents: {len(documents)}")
for i, doc in enumerate(documents):
    print(f"  {i+1}. {doc[:50]}...")



 Étape 4: Préparation des documents de test...
Requête: Tell me about Apple's products
Nombre de documents: 5
  1. Apple is a delicious red fruit that grows on trees...
  2. Apple Inc. is a technology company that makes iPho...
  3. The apple tree produces fruit in autumn seasons an...
  4. Apple's latest iPhone features advanced camera tec...
  5. Green apples are more tart than red apples and con...


In [20]:
#5. Call the reranker
    # What to do: Fill in top_n with how many top results you want returned (e.g., 3).
    # Why: top_n limits the number of reranked results, so you only retrieve the most relevant documents.

import os
from pinecone import Pinecone, RerankModel

pc = Pinecone()

api_key = os.environ.get("PINECONE_API_KEY")

if not api_key:
  print("❌ PINECONE_API_KEY environment variable not set. Cannot perform reranking.")
elif 'pc' not in locals() or not isinstance(pc, Pinecone):
    # This check ensures that the Pinecone client 'pc' was successfully initialized in the previous steps.
    print("❌ Pinecone client 'pc' is not defined or not a valid Pinecone instance. Please ensure the previous steps to initialize the client were successful.")
else:
    # If 'pc' is not defined here, it means the previous step failed.
    # The check above handles this case.
    try:
      reranked = pc.inference.rerank(
          model="bge-reranker-v2-m3",
          query=query,
          documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
          top_n=3
        )
      print("Reranking terminé!")
    except Exception as e:
        print(f"An error occurred during reranking: {e}")

Reranking terminé!


In [51]:
# Étape 6: Inspection des résultats rerankés
    # What to do: Replace ... with code that prints out the rank (i+1), the similarity score m.score, and the document text m.document.text.
    # Why: Seeing these values demonstrates how the reranker orders documents and what scores it assigns.

from pinecone import Pinecone, RerankModel
print("\n Étape 6: Analyse des résultats...")

# Updated function to accept 'results' and iterate over them
def show_reranked(query, results):
    print(f"\n Requête: {query}")
    print(" Résultats rerankés:")
    for i, m in enumerate(results):
        # Assuming 'm' is a RerankResult object with 'document' and 'score'
        print(f"  Rang {i+1}: Score {m.score:.4f} - Document ID: {m.document.id}")
        # The text used for reranking is in m.document.text
        print(f"    Text used for reranking: {m.document.text}")
        print("-" * 50)


    # Analyse pédagogique
    print("\n Analyse:")
    scores = [m.score for m in results] # Use results here
    if scores: # Add check to avoid division by zero if no results
        avg_score = sum(scores) / len(scores)
        print(f"  • Score moyen: {avg_score:.4f}")
        print(f"  • Meilleur score: {max(scores):.4f}")
        # Calculate standard deviation if there is more than one result
        if len(scores) > 1:
            std_dev = (sum([(s - avg_score)**2 for s in scores]) / len(scores))**0.5
            print(f"  • Écart type: {std_dev:.4f}")
        else:
            print("  • Écart type: N/A (only one result)")
    else:
        print("  • Aucune donnée de score à analyser.")


 Étape 6: Analyse des résultats...


Part 2: Setup a Serverless Index for Medical Notes
1. Install data & model libraries

In [52]:
# =============================================================================
# PARTIE 2: Setup a Serverless Index for Medical Notes
# =============================================================================

# Étape 1 : Install data & model libraries
  # What to do: Install pandas for data manipulation, torch for model inference, and transformers for loading embedding models.
  # Why: You’ll use these libraries to load, embed, and manipulate medical note data.

!pip install pandas torch transformers

import os
from pinecone import RerankModel, Pinecone

# Étape 2. Import modules & define environment settings

import os, time, pandas as pd, torch
from pinecone import Pinecone, ServerlessSpec

cloud = "aws"        # e.g., "aws" # Remplacez par votre cloud provider
region = "us-east-1"       # e.g., "us-east-1" # Remplacez par votre région
spec = ServerlessSpec(cloud=cloud, region=region)
index_name = "pinecone-reranker"

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"], environment=f"{cloud}-{region}")

  # What to do: Fill in cloud and region with your Pinecone project’s deployment environment. Choose CPU and memory values in ServerlessSpec.
  # Why: You’re configuring a serverless index tailored to your resource requirements and connecting the client in the proper cloud region.

In [54]:
import pinecone
print(pinecone.__version__)

import os
from pinecone import Pinecone, ServerlessSpec

# Get API key from environment variable
api_key = os.environ.get("PINECONE_API_KEY")

if not api_key:
    print("❌ PINECONE_API_KEY environment variable not set. Cannot proceed.")
else:
    try:
        pc = Pinecone(api_key=api_key)
        pc.create_index(
            name="your-index-name", # 🔁 Replace with your desired index name (lowercase)
            dimension=384,          # 🔁 Replace with the dimension of your embeddings
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1") # 🔁 Replace with your cloud and region
        )
        print("Index creation initiated!")
    except Exception as e:
        print(f"An error occurred during index creation: {e}")

6.0.1
An error occurred during index creation: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin,access-control-request-method,access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-01', 'x-cloud-trace-context': 'b48af485808af83ac2c862ca59980e1d', 'date': 'Fri, 18 Jul 2025 15:46:28 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}



In [55]:
# Création du nouvel index
pc.create_index(
    name=index_name,
    dimension=384,  # Dimension pour all-MiniLM-L6-v2
    spec=spec
)
  # What to do: Set dimension equal to your embedding model’s output size (e.g., 384).
  # Why: The index’s dimension must match the embedding vectors you’ll insert, otherwise upserts will fail.

{
    "name": "pinecone-reranker",
    "metric": "cosine",
    "host": "pinecone-reranker-tu2f5ko.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [59]:
# =============================================================================
# PARTIE 3: Load Sample Data
# =============================================================================

print("\n\n PARTIE 3: Chargement des Données Médicales")
print("=" * 60)

# Étape 1: Download & read JSONL (Simulation de téléchargement et lecture JSONL)
print(" Étape 1: Chargement des données...")

# Comme nous n'avons pas le fichier réel, simulons des données médicales
import json
import numpy as np

# Génération de données médicales simulées
def generate_medical_data(n_samples=50):
    """Génère des données médicales simulées pour l'exercice"""
    import random

    conditions = [
        "Patient presents with chest pain and shortness of breath",
        "Lower back pain with radiating symptoms to legs",
        "Headache with nausea and visual disturbances",
        "Abdominal pain in right lower quadrant",
        "Joint stiffness and swelling in hands",
        "Difficulty swallowing and throat pain",
        "Fatigue and unexplained weight loss",
        "Skin rash with itching and inflammation",
        "Dizziness and balance problems",
        "Chronic cough with blood in sputum"
    ]

    departments = ["Emergency", "Cardiology", "Orthopedics", "Neurology", "Gastroenterology"]
    severities = ["Low", "Medium", "High", "Critical"]

    data = []
    for i in range(n_samples):
        # Générer embedding aléatoire (384 dimensions)
        embedding = np.random.randn(384).tolist()

        record = {
            "id": f"note_{i:03d}",
            "embedding": embedding,
            "metadata": {
                "condition": random.choice(conditions),
                "department": random.choice(departments),
                "severity": random.choice(severities),
                "patient_age": random.randint(20, 80),
                "date": f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
            }
        }
        data.append(record)

    return pd.DataFrame(data)

# Génération des données
df = generate_medical_data(50)
print(f" Données générées: {len(df)} notes médicales")



 PARTIE 3: Chargement des Données Médicales
 Étape 1: Chargement des données...
 Données générées: 50 notes médicales


pcsk_26AKbf_EZMcQjAiAeV8hGN2LPMVx26D92VimSd4BjYmBkqitMZiYSpkHe9EfWy18FczYwM

In [60]:
  #2. Preview the DataFrame

print(df.head())

  # What to do: Run this to view the first few rows of the DataFrame.
  # Why: Ensures you have the right columns (e.g., id, embedding, metadata) before upserting.


         id                                          embedding  \
0  note_000  [1.3295811752261568, 2.2373175148956475, 0.622...   
1  note_001  [-0.5048730332752203, 0.6717284264439994, -0.4...   
2  note_002  [0.33128201541520363, 0.8120346623481423, 0.36...   
3  note_003  [1.0680573560041278, 1.6667233592886155, -0.60...   
4  note_004  [1.5706674432658316, -0.14318616247519358, -1....   

                                            metadata  
0  {'condition': 'Chronic cough with blood in spu...  
1  {'condition': 'Headache with nausea and visual...  
2  {'condition': 'Chronic cough with blood in spu...  
3  {'condition': 'Joint stiffness and swelling in...  
4  {'condition': 'Joint stiffness and swelling in...  


In [61]:
# =============================================================================
# PARTIE 4: Upsert Data into the Index
# =============================================================================

# Etape 1. Instantiate index client & upsert
index = pc.Index(index_name)

# Rename the 'embedding' column to 'values' for upsert_from_dataframe
df = df.rename(columns={'embedding': 'values'})

index.upsert_from_dataframe(df)

  # What to do: Create an Index object and call upsert_from_dataframe.
  # Why: This pushes all your note embeddings and metadata into Pinecone for later queries.

sending upsert requests:   0%|          | 0/50 [00:00<?, ?it/s]

{'upserted_count': 50}

In [62]:
# Etape 2. Wait for availability

def is_ready(idx):
   stats = idx.describe_index_stats()
   return stats.total_vector_count > 0

while not is_ready(index):
   time.sleep(5)
print(index.describe_index_stats())

  # What to do: Poll until total_vector_count is greater than zero.
  # Why: Ensures that upserted vectors are fully indexed before you attempt to query.

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 50}},
 'total_vector_count': 50,
 'vector_type': 'dense'}


In [63]:
# =============================================================================
# PARTIE 5: Query & Embedding Function
# =============================================================================

from sentence_transformers import SentenceTransformer

# Étape 1: Define your embedding function
print(" Étape 1: Configuration du modèle d'embedding...")

from sentence_transformers import SentenceTransformer

def get_embedding(text):
   model = SentenceTransformer("all-MiniLM-L6-v2")  # e.g., "all-MiniLM-L6-v2"
   return model.encode(text)

print("\n\n PARTIE 5: Recherche Sémantique")
print("=" * 60)

  # What to do: Provide the name of the sentence-transformer model you plan to use in place of ....
  # Why: Converts incoming queries into the same vector space as your indexed notes.

 Étape 1: Configuration du modèle d'embedding...


 PARTIE 5: Recherche Sémantique


In [64]:
# Étape 2: Run a semantic search query

print("\n Étape 2: Recherche sémantique...")

question = "What if my patient has leg pain and difficulty walking?"
print(f"Question: {question}")

emb = get_embedding(question)
results = index.query(vector=emb.tolist(), top_k=5, include_metadata=True)
matches = sorted(results.matches, key=lambda m: m.score, reverse=True)

print(f" Recherche terminée! {len(matches)} résultats trouvés")

  # What to do: Replace question, set top_k for number of results (e.g., 5).
  # Why: Retrieves the most semantically similar notes from the index based on your clinical query.


 Étape 2: Recherche sémantique...
Question: What if my patient has leg pain and difficulty walking?
 Recherche terminée! 5 résultats trouvés


In [67]:
# =============================================================================
# PARTIE 6: Display & Rerank Clinical Notes
# =============================================================================

print("\n\n PARTIE 6: Reranking des Notes Cliniques")
print("=" * 60)

# Étape 1: Display initial search results
print(" Étape 1: Résultats de recherche initiaux...")

def show_results(q, matches):
    print(f"\n Question: {q}")
    print(" Résultats initiaux:")
    for i, m in enumerate(matches):
        print(f"  Rang {i+1}: ID={m.id}, Score={m.score:.4f}")
        print(f"    Condition: {m.metadata.get('condition', 'N/A')}")
        print(f"    Département: {m.metadata.get('department', 'N/A')}")
        print(f"    Sévérité: {m.metadata.get('severity', 'N/A')}")
        print("-" * 50)

show_results(question, matches)

  # What to do: Fill in the print statement to show rank, vector ID, similarity score, and metadata.
  # Why: Helps you see which notes were initially considered most relevant.



 PARTIE 6: Reranking des Notes Cliniques
 Étape 1: Résultats de recherche initiaux...

 Question: What if my patient has leg pain and difficulty walking?
 Résultats initiaux:
  Rang 1: ID=note_044, Score=0.0813
    Condition: Joint stiffness and swelling in hands
    Département: Orthopedics
    Sévérité: Critical
--------------------------------------------------
  Rang 2: ID=note_012, Score=0.0749
    Condition: Fatigue and unexplained weight loss
    Département: Neurology
    Sévérité: Medium
--------------------------------------------------
  Rang 3: ID=note_008, Score=0.0677
    Condition: Chronic cough with blood in sputum
    Département: Cardiology
    Sévérité: High
--------------------------------------------------
  Rang 4: ID=note_033, Score=0.0622
    Condition: Abdominal pain in right lower quadrant
    Département: Emergency
    Sévérité: Medium
--------------------------------------------------
  Rang 5: ID=note_041, Score=0.0564
    Condition: Skin rash with itchin

In [68]:
# Étape 2: Prepare documents for reranking
print("\n Étape 2: Préparation pour le reranking...")

rerank_docs = [
    {
        "id": m.id,
        "text": "; ".join([f"{k}: {v}" for k, v in m.metadata.items()])
    }
    for m in matches
]

rerank_query = "Patient presenting with lower extremity pain affecting mobility and walking ability"
print(f" Requête affinée: {rerank_query}")

  # What to do: Set rerank_query to a refined question that tests finer distinctions (e.g., focusing on a procedure or symptom).
  # Why: Constructs a field summarizing each note’s metadata for the reranker to use when rescoring.



 Étape 2: Préparation pour le reranking...
 Requête affinée: Patient presenting with lower extremity pain affecting mobility and walking ability


In [69]:
# Étape 3:  Execute serverless reranking
print("\n Étape 3: Exécution du reranking...")

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=rerank_query,
    documents=rerank_docs,
    top_n=3
)

print(" Reranking terminé!")

  # What to do: Choose top_n to specify how many reranked results you need.
  # Why: Reranking uses the refined query and metadata field to reorder notes by their new relevance scores.


 Étape 3: Exécution du reranking...
 Reranking terminé!


In [72]:
# Étape 4: Show reranked results
print("\n Étape 4: Analyse des résultats rerankés...")

# Updated function call to pass reranked.results
# Also updating the print logic within the function definition (in cell gxlb0pVjtnKl)
# to match the structure of reranked results.
def show_reranked(q, results): # Changed matches to results
   print(f"Refined Query: {q}")
   if results: # Add a check for results
       print(" Résultats rerankés:")
       for i, m in enumerate(results): # Changed matches to results
           # Assuming 'm' is a RerankResult object with 'document' and 'score'
           print(f"  Rang {i+1}: ID={m.document.id}, Score={m.score:.4f}")
           # The text used for reranking is in m.document.text
           print(f"    Reranking Text: {m.document.text}")
   else:
       print(" Aucun résultat de reranking trouvé.") # Message if no results

show_reranked(rerank_query, reranked.results) # Changed reranked.matches to reranked.results

  # What to do: Complete the print logic to display each reranked note’s rank, ID, score, and the reranking_field.
  # Why: Allows you to compare how the reranker improves result ordering against the original search.


 Étape 4: Analyse des résultats rerankés...
Refined Query: Patient presenting with lower extremity pain affecting mobility and walking ability
 Aucun résultat de reranking trouvé.
